# Implementing DICE paper `Methods for Numeracy-Preserving Word Embeddings` by Sundaraman et al

## 1. Motivation and Problem Statement

### 1.1 The Core Problem
Most popular word embedding models like **word2vec** and **GloVe** to contextual models like BERT are built on the distributional hypothesis. This hypothesis states that a word's meaning is defined by the company it keeps. This works beautifully for semantic concepts (e.g., "cat" and "kitten" appear in similar contexts) but it fails spectacularly for numbers.<br>

It happens basically due to the fundamental properties of numbers. The numbers *3* and *4* might appear in very similar contexts ("I have 3 apples," "I have 4 apples"). Based on context alone a model would learn that their embeddings should be very close.

1. **Magnitude:** 4 is greater than 3.
2. **Numeration:** 3 is the same as "three".

This failure is a major issue and it causes models to fail at any task requiring numerical reasoning such as question answering, sentence similarity and data-intensive tasks like machine translation or mathematical operations.

- **Problem Statement:** The authors state that existing word embedding models are ineffective at capturing the numeric properties of numbers. They treat numbers like any other arbitrary word token which leads to unintuitive similarities and a failure to encode magnitude. The goal is to create a new method for number embeddings that explicitly and systematically captures these numerical properties.


## 2. Assumptions and Goals

### 2.1 Assumptions
The paper's central hypothesis or assumption is that a superior numerical embedding can be created if its structure we are not learning from a corpus but instead building deterministically. Specifically authors propose that the cosine similarity between two number embeddings should directly reflect their actual distance on the number line.

### 2.2 Goals
The primary goal is to develop a method to assign and learn embeddings for numbers that correctly capture their numerical properties. Also we want following goals accomplished:
* Provide embeddings $e(x)$ such that cosine distance $d_{\text{cos}}(e(x), e(y))$ monotonically increases with $|x-y|$.
* Keep numeral and word-form representations identical (word tokens that denote numbers point to the numeral embedding).
* Offer a regularizer usable during contextual fine-tuning that encourages similar behavior for learned contextual embeddings.


## 3. Mathematical and Theoretical Concepts

The paper presents two key mathematical concepts: the DICE embedding itself and the regularization loss for contextual models.

### 3.1 DICE: Deterministic Independent of Corpus Embeddings

The core idea of DICE is to map the distance between numbers to the cosine distance between their embedding vectors.

1. **Number Distance ($d_n$):** In the token space or the real number line the distance between two numbers $x$ and $y$ is their absolute difference i.e. $d_n(x, y) = |x - y|$

2. **Embedding Distance $(d_e​)$:** In the embedding space, the distance between two vectors x and y is their cosine distance which is based on the angle θ between them i.e. $d_e(\mathbf{x}, \mathbf{y}) = 1 - \cos(\theta) = 1 - \frac{\mathbf{x}^T \mathbf{y}}{\|\mathbf{x}\|_2 \|\mathbf{y}\|_2}$

3. **Goal:** We want $d_e$ to increase as $d_n$ increases.

#### Algorithm:
```bash
Inputs: numeric range [a, b], embedding dimension D, set of numbers S (subset of [a,b])
1. For each number x in S:
    1.1 θ = (x - a) / |a - b| * π #maps x to [0, π]
    1.2 Construct v ∈ R^D using spherical coords (Eq.6) with angle θ
2. Sample M ∈ R^{D×D} with iid N(0,1) entries
3. QR-decompose M = Q R  (Q orthonormal basis)
4. For each v, the final embedding e(x) = Q * v #random rotation/projection
5. Normalize embeddings (optional: unit norm)
6. For any word token that spells the number (e.g., "three"), point its embedding to same vector e(3).
```

#### Explanation:
The authors achieve this by mapping each individual number $x$ to a specific vector $\mathbf{e}_x$.
- **Angle Mapping:** First authors define a range of numbers they care about $[a, b]$. They linearly map any number $x$ in this range to an angle $\theta(x)$ between $0$ and $\pi$. A number is thus defined by this angle.

- **Vector Generation:** This single angle $\theta$ is then used to generate a $D$-dimensional unit vector $\mathbf{v}$ using a $D$-dimensional polar to cartesian (hyperspherical) coordinate transformation.

- **Random Rotation:** This vector $\mathbf{v}$ is then rotated in the $D$-dimensional space. This is done by multiplying it by a fixed random $D \times D$ orthonormal matrix $\mathbf{Q}$ (which is obtained from a QR decomposition of a random matrix $\mathbf{M}$).

- **Final Embedding:** The final embedding for number $x$ is $\mathbf{e}_x = \mathbf{Q}\mathbf{v}(x)$. This embedding is deterministic (once $\mathbf{Q}$ is fixed) and corpus-independent (it only depends on the value of $x$).

#### Example:
Let $[a,b]=[0,100]$ and $(D=2)$

* For $x=25$: $\theta(25) = 25/100 \cdot \pi = 0.25\pi = 45^\circ$.<br>

  Embedding in 2D: $[ \cos(45^\circ), \sin(45^\circ) ] = [\tfrac{\sqrt2}{2}, \tfrac{\sqrt2}{2}]$.

* For $y=75$: $\theta(75) = 0.75\pi = 135^\circ)$ <br>

  Embedding: $[-\tfrac{\sqrt2}{2}, \tfrac{\sqrt2}{2}]$

* Angle between them is $90^\circ$ and cosine similarity is 0 with cosine distance 1 which tells us that they are mid-range apart $|25−75|=50$.

### 3.2 $ \mathcal{L}_{num}$: Model Based Numeracy Regularization

For contextual models like BERT the authors propose an auxiliary loss function, $\mathcal{L}_{num}$ which is to be added during fine-tuning.<br>
The loss forces the model to learn numerically aware embeddings in context. For any two number embeddings $\mathbf{x}$ and $\mathbf{y}$ taken from the model's final hidden layer then the loss is $\mathcal{L}_{num} = \left\| d_{target} - d_{cos}(\mathbf{x}, \mathbf{y}) \right\|_2^2$

- $\mathbf{x}, \mathbf{y}$ are the contextual embeddings for numbers $x$ and $y$.

- $d_{cos}(\mathbf{x}, \mathbf{y})$ is their cosine distance.

- $d_{target} = \frac{2 |x-y|}{|x|+|y|}$ is the target distance which is a normalized numerical distance between $x$ and $y$ which is always between 0 and 1.

This loss function teaches BERT that the cosine distance between its embeddings for **5** and **10** should be smaller than the distance between its embeddings for **5** and **100**.

## 4. Implementation in Python

### 4.1 Importing Dependencies 

In [1]:
import re
import os
import h5py
import json
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt
from collections import defaultdict

from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from typing import List, Tuple, Union, Dict, Optional 
from scipy.spatial.distance import cosine as cosine_dist
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity



warnings.filterwarnings('ignore')
np.random.seed(42)

### 4.2 Core DICE Implementation

In [ ]:
class DICEEmbeddings:
    
    """
    Here we define the DICEEmbeddings class which will handle the creation of DICE embeddings.
    dimension: Dimension of the embedding space.
    number_range: Range of numbers to be embedded.
    random_rotation: Whether to apply random rotation to the embeddings.
    seed: Random seed for reproducibility.
    """
    def __init__(self, dimension: int = 300, number_range: Tuple[float, float] = (0, 10000),
                 random_rotation: bool = True, seed: int = 42):
        self.dimension = dimension
        self.a, self.b = number_range# a and b are the min and max numbers
        self.range_size = self.b - self.a# Range size
        self.seed = seed
        
        #preventing division by zero
        if self.range_size == 0:
            self.range_size = 1e-9# Small value to prevent division by zero
        
        self.random_rotation = random_rotation# Whether to apply random rotation
        #generating random rotation matrix using QR decomposition as described in the paper
        if self.random_rotation:
            np.random.seed(self.seed)# Seed for reproducibility
            random_matrix = np.random.randn(self.dimension, self.dimension)# Random matrix
            self.rotation_matrix, _ = np.linalg.qr(random_matrix)# QR decomposition to get orthogonal matrix where _ means we ignore the second output
        else:
            self.rotation_matrix = np.eye(self.dimension)# Identity matrix if no rotation
        
        self._embedding_cache = {}# Cache for embeddings to speed up repeated computations
    
    """
    this function computes the spherical coordinates for a given angle theta.
    theta: Angle in radians.
    returns: Numpy array of spherical coordinates.
    v_d = [sin(θ)]^(d-1) * cos(θ)  for 1 ≤ d < D
    v_D = [sin(θ)]^D               for d = D
    """
    def _spherical_coordinates(self, theta:float) -> np.ndarray:
        v = np.zeros(self.dimension)# initialising vector
        sin_theta = np.sin(theta)# sin(θ)
        
        sin_power = 1.0# sin(θ)^(d-1) for avoidance of repeated computation because sin(θ) is used multiple times
        for d in range(1, self.dimension+1):
            if d < self.dimension:# d < D
                # v_d = [sin(θ)]^(d-1) * cos(θ)
                v[d-1] = sin_power * np.cos(theta)
                sin_power *= sin_theta# updating sin(θ)^(d-1) to sin(θ)^d for next iteration
            else:# d = D
                # v_D = [sin(θ)]^D
                v[d-1] = sin_power * sin_theta# sin(θ)^D
        return v
    
    """
    This function maps a number to an angle in radians within the range [0, π/2].
    θ(s_n) = (s_n / |a - b|) * π
    where s_n is the distance from the lower bound a.
    number: The number to be mapped.
    returns: Angle in radians.
    """
    def _number_to_angle(self, number: float) -> float:
        if number < self.a or number > self.b:
            np.random.seed(int(hash(number) % (2**32)))
            theta = np.random.uniform(-np.pi, np.pi)
        else:
            # Linear mapping from [a, b] to [0, π]
            normalized_distance = (number - self.a) / self.range_size
            theta = normalized_distance * np.pi

        return theta
    
    """
    This function computes the DICE embedding for a given number.
    number: The number to be embedded.
    use_cache: Whether to use cached embeddings.
    returns: Numpy array of the DICE embedding.
    """
    def get_embedding(self, number: Union[int, float], use_cache: bool = True) -> np.ndarray:
        # Check cache
        if use_cache and number in self._embedding_cache:
            return self._embedding_cache[number]
        
        # Map number to angle
        theta = self._number_to_angle(number)
        
        # Get spherical coordinate vector
        v = self._spherical_coordinates(theta)
        
        # Apply random rotation (Q @ v)
        embedding = self.rotation_matrix @ v
        
        # Cache the result
        if use_cache:
            self._embedding_cache[number] = embedding
        
        return embedding
    
    """
    This function computes DICE embeddings for a batch of numbers.
    numbers: List of numbers to be embedded.
    use_cache: Whether to use cached embeddings.
    returns: Numpy array of DICE embeddings.
    """
    def get_embeddings_batch(self, numbers: List[Union[int, float]], 
                            use_cache: bool = True) -> np.ndarray:
        return np.array([self.get_embedding(num, use_cache) for num in numbers])
    
    def clear_cache(self):# Clear the embedding cache
        self._embedding_cache = {}
    
    # This function retrieves the angle corresponding to a given number.
    def get_angle(self, number: Union[int, float]) -> float:
        return self._number_to_angle(number)

### 4.3 Data Extraction and Pre-Processing

In [3]:
"""
This function extracts numbers from a nested JSON structure recursively.
data: The JSON data (can be dict, list, or primitive).
returns: List of extracted numbers.
"""
def extract_numbers_from_json(data) -> List[float]:
    numbers = []# List to store extracted numbers
    if isinstance(data,dict):# if data is a dictionary
        for k,v in data.items():# iterate through key-value pairs
            numbers.extend(extract_numbers_from_json(v))# recursively extract numbers from value
    elif isinstance(data,list):# if data is a list
        for item in data:# iterate through items
            numbers.extend(extract_numbers_from_json(item))# recursively extract numbers from item
    elif isinstance(data, (str, int,float)):# if data is a primitive type
        try:
            s = str(data)
            # Regex to find numbers: integers, floats, scientific notation
            found_nums = re.findall(r'-?\d+\.?\d*(?:[eE][+-]?\d+)?', s)
            for n in found_nums:
                try:
                    numbers.append(float(n))
                except ValueError:
                    pass
        except (ValueError, TypeError):
            pass
    return numbers

In [4]:
def parse_numbers_from_text(text: str) -> List[float]:
    pattern = r'-?\d+\.?\d*(?:[eE][+-]?\d+)?'
    matches = re.findall(pattern, text)
    return [float(m) for m in matches]# returns list of floats for given text

### 4.4: Performing Experiments and Tests(Section4.1 of Paper)

In [6]:
"""
This class implements first experiment from the paper.
We will test numeration(NUM) and magnitude(MAG) properties using:
1. one vs all(ova)
2. strict contrastive(sc)
3. broad contrastive(bc)
as described in the paper.
"""
class NumerationMagnitudeTests:
    
    def __init__(self, dice_model: DICEEmbeddings):
        self.dice_model = dice_model
    
    """
    This function finds the k nearest neighbors of a target embedding using cosine distance.
    target_embedding: The embedding for which to find neighbors.
    embeddings: Array of candidate embeddings.
    k: Number of neighbors to find.
    returns: Indices of the k nearest neighbors.
    """    
    def _get_nearest_neighbors(self, target_embedding:np.ndarray,
                               embeddings: np.ndarray,
                               k:int = None) -> np.ndarray:
        #compute cosine distances
        distances = cosine_distances(target_embedding.reshape(1,-1), embeddings)[0]# reshape to 1D array
        indices = np.argsort(distances)# sort distances to get indices of nearest neighbors
        return indices if k is None else indices[:k]# return all or top k indices
    
    """
    This function performs the one-vs-all test for numeration or magnitude.
    numbers: List of numbers to test.
    test_type: 'NUM' for numeration, 'MAG' for magnitude.
    returns: Accuracy of the test.
    """
    def test_one_vs_all(self, numbers:List[float],
                        test_type: str = 'MAG') -> float:
        passed = 0# count of passed tests
        total = 0# total tests
        
        for i, target in enumerate(numbers):
            target_emb = self.dice_model.get_embedding(target)# get embedding for target number
            other_numbers = numbers[:i] + numbers[i+1:]# all other numbers
            
            if len(other_numbers)<2:
                continue# need at least 2 other numbers to compare
            other_embs = self.dice_model.get_embeddings_batch(other_numbers)# get embeddings for other numbers
            neighbors = self._get_nearest_neighbors(target_emb, other_embs, k=2)# get 2 nearest neighbors
            #computing distances
            dist_nearest = cosine_dist(target_emb, other_embs[neighbors[0]])# distance to nearest neighbor
            dist_second = cosine_dist(target_emb, other_embs[neighbors[1]])# distance to second nearest neighbor
            
            if dist_nearest < dist_second:
                passed += 1# test passed
            total += 1# increment total tests
        
        return (passed / total) *100 if total > 0 else 0.0# return accuracy percentage

    """
    This function performs the broad contrastive test for numeration or magnitude.
    numbers: List of numbers to test.
    test_type: 'NUM' for numeration, 'MAG' for magnitude.
    returns: Accuracy of the test."""   
    def test_broad_contrastive(self, numbers: List[float],
                               test_type: str = 'MAG') -> float:
        passed = 0
        total = 0
        
        for i, target in enumerate(numbers):
            target_emb = self.dice_model.get_embedding(target)
            other_numbers = numbers[:i] + numbers[i+1:]# all other numbers
            if len(other_numbers)<2:
                continue# need at least 2 other numbers to compare
            other_embs = self.dice_model.get_embeddings_batch(other_numbers)
            #getting nearest and farthest neighbors
            all_neighbors = self._get_nearest_neighbors(target_emb, other_embs)
            nearest_idx = all_neighbors[0]# nearest neighbor index
            farthest_idx = all_neighbors[-1]# farthest neighbor index -1 because last index
            
            # computing distances
            dist_nearest = cosine_dist(target_emb, other_embs[nearest_idx])# distance to nearest neighbor
            dist_farthest = cosine_dist(target_emb, other_embs[farthest_idx])# distance to farthest neighbor
            if dist_nearest < dist_farthest:
                passed += 1# test passed
            total += 1# increment total tests
        return (passed / total) *100 if total > 0 else 0.0# return accuracy percentage
    
    
    def run_all_tests(self, numbers: List[float]) -> Dict[str, float]:
        print("\n" + "="*35)
        print("TASK 1: Numeration AND Magnitude Tests")
        print("="*35)

        results = {}# Dictionary to store results

        # Running MAG tests
        print("\nMagnitude (MAG) Tests")
        results['OVA-MAG'] = self.test_one_vs_all(numbers, 'MAG')
        print(f"OVA-MAG: {results['OVA-MAG']:.2f}%")

        results['SC-MAG'] = self.test_strict_contrastive(numbers, 'MAG')
        print(f"SC-MAG:  {results['SC-MAG']:.2f}%")

        results['BC-MAG'] = self.test_broad_contrastive(numbers, 'MAG')
        print(f"BC-MAG:  {results['BC-MAG']:.2f}%")

        # Running NUM tests (simplified since we don't have word forms)
        print("\nNumeration (NUM) Tests")
        return results

### 4.5 Listing Maximum, Decoding and Addition of Two Numbers

In [7]:
"""
Implementation of Task 2 from the paper (Section 4.2):
- List Maximum: Find max from 5 numbers
- Decoding: Regress number from embedding
- Addition: Predict sum of two numbers
"""
class NumericalReasoningTests:
    def __init__(self, dice_model: DICEEmbeddings):
        self.dice_model = dice_model# DICE embeddings model

    def generate_list_maximum_data(self, num_range: Tuple[int, int],
                                   n_samples: int = 10000) -> Tuple[np.ndarray, np.ndarray]:
        min_val, max_val = num_range# min and max values
        X_lists = []# List to store embeddings
        y_labels = []# List to store labels (index of max number)
        for _ in range(n_samples):
            base = np.random.randint(min_val, max_val - 10)# base number
            numbers = np.random.randint(base, base + 10, size=5)# generate 5 numbers within range
            max_idx = np.argmax(numbers)# index of maximum number

            # Getting embeddings
            embeddings = self.dice_model.get_embeddings_batch(numbers.tolist())
            X_lists.append(embeddings)# append embeddings
            y_labels.append(max_idx)# append label

        return np.array(X_lists), np.array(y_labels)

    def test_list_maximum(self, num_range: Tuple[int, int]) -> float:
        print(f"\nList Maximum Test: Range {num_range}")

        # Generate data
        X_train, y_train = self.generate_list_maximum_data(
            num_range, n_samples=8000)
        X_test, y_test = self.generate_list_maximum_data(
            num_range, n_samples=2000)

        # Flatten embeddings for simple approach
        X_train_flat = X_train.reshape(X_train.shape[0], -1)
        X_test_flat = X_test.reshape(X_test.shape[0], -1)

        # Use KNN classifier
        knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
        knn.fit(X_train_flat, y_train)
        # Predict
        y_pred = knn.predict(X_test_flat)
        accuracy = np.mean(y_pred == y_test) * 100

        print(f"Accuracy: {accuracy:.2f}%")
        return accuracy

    def test_decoding(self, num_range: Tuple[int, int]) -> Dict[str, float]:
        print(f"\nDecoding Test: Range {num_range}")

        # Generate numbers
        min_val, max_val = num_range
        numbers = list(range(min_val, max_val + 1))

        # Get embeddings
        X = self.dice_model.get_embeddings_batch(numbers)
        y = np.array(numbers)

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Train KNN regressor
        knn = KNeighborsRegressor(n_neighbors=5, metric='cosine')
        knn.fit(X_train, y_train)

        # Predict
        y_pred = knn.predict(X_test)

        # Compute metrics
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)

        print(f"RMSE: {rmse:.2f}")
        print(f"MAE: {mae:.2f}")

        return {'rmse': rmse, 'mae': mae, 'y_test': y_test, 'y_pred': y_pred}

    def test_addition(self, num_range: Tuple[int, int]) -> Dict[str, float]:
        print(f"\nAddition Test: Range {num_range}")
        min_val, max_val = num_range
        # Generate training data
        n_samples = 10000
        X1_list, X2_list, y_list = [], [], []

        for _ in range(n_samples):
            num1 = np.random.randint(min_val, max_val + 1)# first number
            num2 = np.random.randint(min_val, max_val + 1)# second number
            total = num1 + num2# sum of the two numbers

            emb1 = self.dice_model.get_embedding(num1)# get embedding for first number
            emb2 = self.dice_model.get_embedding(num2)# get embedding for second number
            X1_list.append(emb1)# append first embedding
            X2_list.append(emb2)# append second embedding
            y_list.append(total)# append sum

        X1 = np.array(X1_list)# first embeddings
        X2 = np.array(X2_list)# second embeddings
        y = np.array(y_list)# sums

        # Concatenate embeddings
        X = np.concatenate([X1, X2], axis=1)

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Train simple feedforward network using sklearn
        mlp = MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=100,
                           random_state=42, verbose=False)
        mlp.fit(X_train, y_train)

        # Predict
        y_pred = mlp.predict(X_test)

        # Compute metrics
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)

        print(f"RMSE: {rmse:.2f}")
        print(f"MAE: {mae:.2f}")

        return {'rmse': rmse, 'mae': mae}

### 4.6 Visualisation of Results

In [8]:
def visualize_dice_2d_projection(dice_model: DICEEmbeddings,
                                 numbers: List[float],
                                 save_path: str = 'dice_2d_projection.png',
                                 drive_path: str = None):
    # Get embeddings
    embeddings = dice_model.get_embeddings_batch(numbers)

    # Project to 2D
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(embeddings)
    # Create visualization
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    # Color by magnitude
    scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
                         c=numbers, cmap='viridis', s=100, alpha=0.7)
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Number Value', fontsize=12)# label for colorbar
    step = max(1, len(numbers) // 20)# step for annotations
    for i in range(0, len(numbers), step):
        ax.annotate(f'{numbers[i]:.0f}',
                    (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                    fontsize=8, alpha=0.6)

    ax.set_xlabel(
        f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
    ax.set_ylabel(
        f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
    ax.set_title('DICE Embeddings - 2D PCA Projection',
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    if drive_path:
        save_path = os.path.join(drive_path, 'visualizations', save_path)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved 2D projection to '{save_path}'")
    plt.show()
    plt.close()

In [9]:
def visualize_cosine_correlation(dice_model: DICEEmbeddings, 
                                test_numbers: List[float],
                                save_path: str = 'dice_correlation_analysis.png',
                                drive_path: str = None):
    print("\n" + "="*50)
    print("CORRELATION ANALYSIS: Cosine Distance vs Numerical Distance")
    print("="*50)
    # Get embeddings
    embeddings = dice_model.get_embeddings_batch(test_numbers)
    # Compute distances
    cosine_dists = cosine_distances(embeddings)
    test_numbers_arr = np.array(test_numbers)
    numerical_dists = np.abs(test_numbers_arr[:, None] - test_numbers_arr)
    # Create figure with multiple subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    # Plot 1: Scatter plot
    mask = np.triu_indices_from(cosine_dists, k=1)
    cosine_flat = cosine_dists[mask]# flattened upper triangle
    numerical_flat = numerical_dists[mask]
    # Remove NaN or inf values
    valid_mask = np.isfinite(numerical_flat) & np.isfinite(cosine_flat)
    cosine_flat = cosine_flat[valid_mask]
    numerical_flat = numerical_flat[valid_mask]
    
    axes[0, 0].scatter(numerical_flat, cosine_flat, alpha=0.1, s=1)
    axes[0, 0].set_xlabel('Numerical Distance |x-y|', fontsize=12)
    axes[0, 0].set_ylabel('Cosine Distance', fontsize=12)
    axes[0, 0].set_title('Cosine vs Numerical Distance', fontsize=14, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Calculate and display correlation
    if len(cosine_flat) > 0:
        correlation = np.corrcoef(numerical_flat, cosine_flat)[0, 1]
        axes[0, 0].text(0.05, 0.95, f'Correlation: {correlation:.4f}', 
                       transform=axes[0, 0].transAxes, fontsize=12,
                       bbox=dict(boxstyle="round,pad=0.5", facecolor="yellow", alpha=0.7))
        print(f"Correlation coefficient: {correlation:.4f}")
    
    # Plot 2: Cosine distance matrix
    im = axes[0, 1].imshow(cosine_dists, cmap='viridis', aspect='auto')
    axes[0, 1].set_title('Cosine Distance Matrix', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Number Index (Sorted)', fontsize=12)
    axes[0, 1].set_ylabel('Number Index (Sorted)', fontsize=12)
    plt.colorbar(im, ax=axes[0, 1], label='Cosine Distance')
    
    # Plot 3: Numerical distance matrix
    im2 = axes[1, 0].imshow(numerical_dists, cmap='viridis', aspect='auto')
    axes[1, 0].set_title('Numerical Distance Matrix', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Number Index (Sorted)', fontsize=12)
    axes[1, 0].set_ylabel('Number Index (Sorted)', fontsize=12)
    plt.colorbar(im2, ax=axes[1, 0], label='Numerical Distance')
    
    # Plot 4: Hexbin density plot
    axes[1, 1].hexbin(numerical_flat, cosine_flat, gridsize=50, cmap='YlOrRd', mincnt=1)
    axes[1, 1].set_xlabel('Numerical Distance |x-y|', fontsize=12)
    axes[1, 1].set_ylabel('Cosine Distance', fontsize=12)
    axes[1, 1].set_title('Density Plot (Hexbin)', fontsize=14, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Save to appropriate location
    if drive_path:
        save_path = os.path.join(drive_path, 'visualizations', save_path)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved correlation analysis to '{save_path}'")
    plt.show()
    plt.close()

In [12]:
def visualize_decoding_performance(y_test: np.ndarray, 
                                  y_pred: np.ndarray,
                                  dimension: int,
                                  num_range: Tuple[int, int],
                                  save_path: str = 'dice_decoding_performance.png',
                                  drive_path: str = None):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    # Plot 1Scatter plot with ideal line
    axes[0].scatter(y_test, y_pred, alpha=0.5, s=20)
    # Add ideal line
    lims = [
        np.min([axes[0].get_xlim(), axes[0].get_ylim()]),
        np.max([axes[0].get_xlim(), axes[0].get_ylim()]),
    ]
    axes[0].plot(lims, lims, 'r--', linewidth=2, label='Ideal (Predicted = True)')
    
    axes[0].set_xlabel("True Number Value", fontsize=14)
    axes[0].set_ylabel("Predicted Number Value", fontsize=14)
    axes[0].set_title(f"DICE Decoding Task (D={dimension}, Range={num_range})", 
                     fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=12)
    axes[0].grid(True, alpha=0.3)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = 1 - (np.sum((y_test - y_pred)**2) / np.sum((y_test - np.mean(y_test))**2))
    
    # Add metrics text
    metrics_text = f'RMSE: {rmse:.2f}\nMAE: {mae:.2f}\nR²: {r2:.4f}'
    axes[0].text(0.05, 0.95, metrics_text, transform=axes[0].transAxes,
                fontsize=11, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Plot 2: Residuals
    residuals = y_test - y_pred
    axes[1].scatter(y_test, residuals, alpha=0.5, s=20)
    axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
    axes[1].set_xlabel("True Number Value", fontsize=14)
    axes[1].set_ylabel("Residual (True - Predicted)", fontsize=14)
    axes[1].set_title("Residual Plot", fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    # Save to appropriate location
    if drive_path:
        save_path = os.path.join(drive_path, 'visualizations', save_path)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved decoding performance plot to '{save_path}'")
    plt.show()
    plt.close()

In [13]:
def visualize_angle_mapping(dice_model: DICEEmbeddings,
                           numbers: List[float],
                           save_path: str = 'dice_angle_mapping.png',
                           drive_path: str = None):
    angles = [dice_model.get_angle(num) for num in numbers]
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    # Plot 1: Linear mapping
    axes[0].plot(numbers, angles, 'b-', linewidth=2)
    axes[0].scatter(numbers, angles, c=numbers, cmap='viridis', s=50, alpha=0.7)
    axes[0].set_xlabel('Number Value', fontsize=14)
    axes[0].set_ylabel('Angle θ (radians)', fontsize=14)
    axes[0].set_title('Number to Angle Mapping (Equation 4)', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
    axes[0].axhline(y=np.pi, color='k', linestyle='--', alpha=0.3)
    
    # Add range indicators
    axes[0].text(dice_model.a, -0.2, f'a={dice_model.a}', ha='center', fontsize=10)
    axes[0].text(dice_model.b, np.pi + 0.2, f'b={dice_model.b}', ha='center', fontsize=10)
    
    # Plot 2: Circular representation
    ax = axes[1]
    ax = plt.subplot(1, 2, 2, projection='polar')
    
    # Plot numbers on unit circle
    scatter = ax.scatter(angles, np.ones_like(angles), c=numbers, 
                        cmap='viridis', s=100, alpha=0.7)
    ax.set_ylim(0, 1.2)
    ax.set_title('Circular Angle Representation', fontsize=14, fontweight='bold', pad=20)
    
    plt.colorbar(scatter, ax=ax, label='Number Value', pad=0.1)
    
    # Save to appropriate location
    if drive_path:
        save_path = os.path.join(drive_path, 'visualizations', save_path)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved angle mapping visualization to '{save_path}'")
    plt.show()
    plt.close()